# Limpieza de Datos – Marine

## 1. Importación de paqueterías 


In [107]:
import os
import pandas as pd
from tabulate import tabulate
from unidecode import unidecode
import re
import numpy as np
from rapidfuzz import fuzz, process
from collections import defaultdict, Counter
import ahocorasick
import pandas as pd
import re
from unidecode import unidecode
from collections import OrderedDict
from opcion1_estado_offline import agregar_estado
# o: from opcion2_estado_geocoding import agregar_estado_hibrido


## 2. Carga de datos

In [108]:
#Carga de archivo original
# RECOMENDADO
import os
BASE_DIR = os.environ.get("MARINE_BASE_DIR", "C:/Users/IKAL14/Documents/Integral/Marine")
PERIODO = os.environ.get("PERIODO", "202512")
archivo_original = f"{BASE_DIR}/Procesados/{PERIODO}/Final/{PERIODO}_Siniestros_Marine.xlsx"

# Cargar archivo Excel o CSV
df = pd.read_excel(archivo_original, dtype={"INWARD POLICY N°": str})   # o pd.read_csv("datos.csv")
#archivoreadme= "C:\Users\IKAL14\OneDrive - Kot Insurance Company AG\Vida\Vida 2025\readme.txt"

### Extracción del nombre del archivo


In [109]:
# Extraer año y mes del nombre del archivo
nombre_base = os.path.basename(archivo_original)  # '202508_Siniestros_Vida.xlsx'
ano_mes = nombre_base[:6]                        # '202508'
ano = ano_mes[:4]                                # '2025'
mes = ano_mes[4:]                                # '08'
mes_num = int(ano_mes[4:])                        # 8 (convertido a entero)

nombre_archivo_salida = f"{ano_mes}_Siniestros_Marine_PROCESADO.xlsx"

# Carpeta y nombre de archivo de salida
#carpeta_salida = f"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/06-RM - 02-Actuarial/6. DATABASE/03 Transporte, Carga & Embarcaciones/02 Bases Actuarial/{ano_mes}"
carpeta_salida2 = f"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/{ano}/{ano_mes}"

# Crear carpeta si no existe
#os.makedirs(carpeta_salida, exist_ok=True)
os.makedirs(carpeta_salida2, exist_ok=True)

# Quitar filas completamente vacías
df = df.dropna(how="all")

#Agregar columna con mes de carga
df["MES CARGA"] = pd.Timestamp(f"{ano}-{mes}")



## 3. Limpieza de datos

### 3.1 Reemplazo y relleno de datos


In [110]:
# ================================
# CONFIGURACIÓN DE MAPEOS (Siempre en MAYÚSCULAS)
# ================================
cover_map = {
    "CARGA": "CARGA POLIETILENO",
    "CARGO": "CARGO",
    "RC FLETADORES": "RC FLETADORES(PEMEX)",
    "Fletadores RC PMI": "RC FLETADORES(PMI)",
    "FLETADORES RC PMI": "RC FLETADORES(PMI)",
    "FLETADORES PMI": "FLETADORES(PMI)",
    "FLETADORESPMI": "FLETADORES(PMI)",
    "CASCO Y MAQUINARIA": "CASCO Y MAQ.",
    "CASCO": "CASCO Y MAQ.",
    "DAÑOS A LA MAQUINARIA": "CASCO Y MAQ.",
    "PANDI": "P&I",
    "JACK-UPS": "JACK-UPS(DAÑO FISICO)",
    "GASTOS DE SALVAMENTO": "CASCO Y MAQ.",
    "TRANSPORTE": "CARGO",
    "AGUAS PROFUNDAS": "DEEP WATERS",
    "DEEP WATERS": "DEEP WATERS",
}

VALID_STATUSES = {"O", "P", "C", "T"}

# ================================
# LIMPIEZA POR COLUMNAS
# ================================

# 1. INWARD POLICY N°
df["INWARD POLICY N°"] = (
    df["INWARD POLICY N°"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)  # Quita .0 de floats
    .str.strip("'")  # Quita comillas previas
)

# 2. DESCRIPTION (Unificado y optimizado sin .apply)
df["DESCRIPTION"] = (
    df["DESCRIPTION"]
    .fillna("NO ESPECIFICADO")
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(
        r"['\"\[\]\(\)\{\}•*“”]", "", regex=True
    )  # Remueve símbolos respetando Ñ, acentos y guiones
)

# 3. COVER
df["COVER"] = (
    df["COVER"]
    .fillna("NO ESPECIFICADO")
    .astype(str)
    .str.replace(
        "[\xa0\u200b\ufeff\t\r]", " ", regex=True
    )  # Caracteres invisibles a espacio
    .str.replace(r"\s+", " ", regex=True)  # Colapsa múltiples espacios
    .str.strip()
    .str.upper()
    .replace(cover_map)  # Mapeo homologado
)

# 4. LoB-Inward (Tratamiento de nulos + Herencia + Reemplazos)
df["LoB-Inward"] = (
    df["LoB-Inward"]
    .fillna(df["COVER"])
    .astype(str)
    .str.strip()
    .str.upper()
)
df.loc[df["LoB-Inward"] == "NAN", "LoB-Inward"] = df["COVER"]
df.loc[df["LoB-Inward"] == "RC FLETADORES(PEMEX)", "COVER"] = "RC FLETADORES(PEMEX)"
df.loc[df["COVER"] == "FLETADORES(PMI)", "LoB-Inward"] = "FLETADORES(PMI)"

df.loc[
    (df["LoB-Inward"] == "CARGA POLIETILENO")
    & (df["COVER"].isin(["CARGO", "Carga"])),
    "COVER"
] = "CARGA POLIETILENO"

mask = (
    df["LoB-Inward"].notna()
    & df["LoB-Inward"].astype(str).str.strip().ne("")
    & df["LoB-Inward"].astype(str).str.strip().ne("NO ESPECIFICADO")
)

df.loc[mask, "COVER"] = df.loc[mask, "LoB-Inward"]
# Aplicar reemplazos específicos en LoB-Inward
df["LoB-Inward"] = df["LoB-Inward"].replace(
    {
        "CARGA": "CARGA POLIETILENO",
        "CARGO": "CARGO",
        "FLETADORES PMI": "RC FLETADORES(PMI)",
    }
)

# 5. OTRAS COLUMNAS (Rellenos fijos de nulos)
df["COVERAGE"] = df["COVERAGE"].fillna("NO ESPECIFICADO")

#Normalizar LOCATION (Unificado y optimizado sin .apply)
def normalizar(texto):
    """Normaliza texto: mayusculas, sin acentos, sin puntuacion ruidosa."""
    if pd.isna(texto) or str(texto).strip() in ("", "-", "nan", "N/A"):
        return ""
    t = unidecode(str(texto)).upper().strip()
    t = re.sub(r"[;:'\"\[\]\(\)\{\}*#!]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["LOCATION"] = df["LOCATION"].fillna("NO ESPECIFICADO")
df.loc[df["LOCATION"] == "-", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "Pendiente", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "pendiente", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "PENDIENTE", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "CD", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "Litigio", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "MANCHAS A UN BUQUE LPG BERING GAS", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "Terminal de Pemex", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "REF.RC-R-204-2010", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "Lugar/Terminal: PACIFIC AREA LIGHTERING", "LOCATION"] = "PACIFICO NORTE"

#Apply normalización a toda la columna LOCATION
df["LOCATION"] = df["LOCATION"].apply(normalizar)

df["CLAIMS-ID"] = df["CLAIMS-ID"].fillna(df["KEY LOB"])

# Eliminación de columnas sobrantes
df = df.drop(["NOTAS", "REF. PEMEX"], axis=1, errors="ignore")

# ================================
# RECLASIFICACIÓN DE STATUS
# ================================
df["STATUS"] = (
    df["STATUS"]
    .fillna("NO ESPECIFICADO")
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({"NAN": "NO ESPECIFICADO"})
)

df["STATUS_ORIGINAL"] = df["STATUS"].copy()

# Máscara para identificar registros sin estatus válido asignado de origen
mask_sin_status = df["STATUS"].isin(["NO ESPECIFICADO", "NAN"]) | df["STATUS"].isna()

# Reglas de negocio para asignación automática
#Status T suposición de Barby
df.loc[mask_sin_status & (df["OSLR Inward"] > 0), "STATUS"] = "T"
df.loc[
    mask_sin_status & (df["OSLR Inward"] == 0) & (df["Cumulative CLAIMS PAID"] > 0),
    "STATUS",
] = "P"
df.loc[
    mask_sin_status & (df["OSLR Inward"] == 0) & (df["Cumulative CLAIMS PAID"] == 0),
    "STATUS",
] = "C"

# Control de calidad / Auditoría
invalidos = df[~df["STATUS"].isin(VALID_STATUSES)]
if not invalidos.empty:
    print(f"ALERTA: {len(invalidos)} registros con STATUS no reconocido")

# ================================
print("Limpieza y reclasificación completada. Resumen de STATUS:")
print(df["STATUS"].value_counts())
# ================================

Limpieza y reclasificación completada. Resumen de STATUS:
STATUS
P    1945
C     778
T     160
Name: count, dtype: int64


In [111]:
print(f"Registros después de limpieza básica: {len(df)}")

Registros después de limpieza básica: 2883


### 3.2 Normalización para Location


In [112]:
import re
from collections import OrderedDict
import pandas as pd
from unidecode import unidecode

# ============================================================
# CATALOGO 1: Estados de Mexico (nombre limpio -> nombre oficial)
# ============================================================
ESTADOS_MX = OrderedDict([
    ("BAJA CALIFORNIA SUR",   "BAJA CALIFORNIA SUR"),
    ("QUINTANA ROO",          "QUINTANA ROO"),
    ("SAN LUIS POTOSI",       "SAN LUIS POTOSI"),
    ("CIUDAD DE MEXICO",      "CIUDAD DE MEXICO"),
    ("ESTADO DE MEXICO",      "ESTADO DE MEXICO"),
    ("BAJA CALIFORNIA",       "BAJA CALIFORNIA"),
    ("NUEVO LEON",            "NUEVO LEON"),
    ("AGUASCALIENTES",        "AGUASCALIENTES"),
    ("CAMPECHE",              "CAMPECHE"),
    ("CHIAPAS",               "CHIAPAS"),
    ("CHIHUAHUA",             "CHIHUAHUA"),
    ("COAHUILA",              "COAHUILA"),
    ("COLIMA",                "COLIMA"),
    ("DURANGO",               "DURANGO"),
    ("GUANAJUATO",            "GUANAJUATO"),
    ("GUERRERO",              "GUERRERO"),
    ("HIDALGO",               "HIDALGO"),
    ("JALISCO",               "JALISCO"),
    ("MICHOACAN",             "MICHOACAN"),
    ("MORELOS",               "MORELOS"),
    ("NAYARIT",               "NAYARIT"),
    ("OAXACA",                "OAXACA"),
    ("PUEBLA",                "PUEBLA"),
    ("QUERETARO",             "QUERETARO"),
    ("SINALOA",               "SINALOA"),
    ("SONORA",                "SONORA"),
    ("TABASCO",               "TABASCO"),
    ("TAMAULIPAS",            "TAMAULIPAS"),
    ("TLAXCALA",              "TLAXCALA"),
    ("VERACRUZ",              "VERACRUZ"),
    ("YUCATAN",               "YUCATAN"),
    ("ZACATECAS",             "ZACATECAS"),
])

# ============================================================
# CATALOGO 2: Abreviaturas -> Estado oficial
# ============================================================
ABREVIATURAS = {
    "AGS":            "AGUASCALIENTES",
    "BC":             "BAJA CALIFORNIA",
    "B.C":            "BAJA CALIFORNIA",
    "B.C.":           "BAJA CALIFORNIA",
    "BCS":            "BAJA CALIFORNIA SUR",
    "B.C.S":          "BAJA CALIFORNIA SUR",
    "CAMP":           "CAMPECHE",
    "CHIS":           "CHIAPAS",
    "CHIH":           "CHIHUAHUA",
    "COAH":           "COAHUILA",
    "COL":            "COLIMA",
    "CDMX":           "CIUDAD DE MEXICO",
    "D.F.":           "CIUDAD DE MEXICO",
    "D.F":            "CIUDAD DE MEXICO",
    "DF":             "CIUDAD DE MEXICO",
    "DGO":            "DURANGO",
    "GTO":            "GUANAJUATO",
    "GRO":            "GUERRERO",
    "HGO":            "HIDALGO",
    "JAL":            "JALISCO",
    "MEX":            "ESTADO DE MEXICO",
    "EDOMEX":         "ESTADO DE MEXICO",
    "EDO MEX":        "ESTADO DE MEXICO",
    "EDO. MEX":       "ESTADO DE MEXICO",
    "EDO DE MEX":     "ESTADO DE MEXICO",
    "EDO. DE MEX":    "ESTADO DE MEXICO",
    "EDO DE MEXICO":  "ESTADO DE MEXICO",
    "EDO. DE MEXICO": "ESTADO DE MEXICO",
    "EDO. MEXICO":    "ESTADO DE MEXICO",
    "MICH":           "MICHOACAN",
    "MOR":            "MORELOS",
    "NAY":            "NAYARIT",
    "NL":             "NUEVO LEON",
    "NL.":            "NUEVO LEON",
    "N.L":            "NUEVO LEON",
    "N.L.":           "NUEVO LEON",
    "OAX":            "OAXACA",
    "PUE":            "PUEBLA",
    "QRO":            "QUERETARO",
    "Q. ROO":         "QUINTANA ROO",
    "QROO":           "QUINTANA ROO",
    "Q.ROO":          "QUINTANA ROO",
    "SLP":            "SAN LUIS POTOSI",
    "SIN":            "SINALOA",
    "SON":            "SONORA",
    "TAB":            "TABASCO",
    "TAM":            "TAMAULIPAS",
    "TAMPS":          "TAMAULIPAS",
    "TLAX":           "TLAXCALA",
    "VER":            "VERACRUZ",
    "VER.":           "VERACRUZ",
    "VERACRZ":        "VERACRUZ",
    "YUC":            "YUCATAN",
    "ZAC":            "ZACATECAS",
}

# ============================================================
# CATALOGO 3: Ciudades (Incluye parches para la imagen)
# ============================================================
CIUDADES = {
    # Tamaulipas
    "CIUDAD MADERO":      "TAMAULIPAS",
    "CD MADERO":          "TAMAULIPAS",
    "CD. MADERO":         "TAMAULIPAS",
    "ALTAMIRA":           "TAMAULIPAS",
    "TAMPICO":            "TAMAULIPAS",
    "REYNOSA":            "TAMAULIPAS",
    "MATAMOROS":          "TAMAULIPAS",
    "CIUDAD VICTORIA":    "TAMAULIPAS",
    "CD VICTORIA":        "TAMAULIPAS",
    "CD. VICTORIA":       "TAMAULIPAS",
    "NUEVO LAREDO":       "TAMAULIPAS",
    "MANTE":              "TAMAULIPAS",
    "CIUDAD MANTE":       "TAMAULIPAS",
    # Veracruz
    "COATZACOALCOS":      "VERACRUZ",
    "COATZA":             "VERACRUZ", 
    "COATZACOALCO":       "VERACRUZ",
    "COATZACOALCOS ":     "VERACRUZ",
    "COTZCOALCOS":        "VERACRUZ", 
    "COTZCOA":            "VERACRUZ", # <-- Parche para el recorte "COTZCOA..." de la imagen
    "MINATITLAN":         "VERACRUZ",
    "POZA RICA":          "VERACRUZ",
    "TUXPAN":             "VERACRUZ",
    "CORDOBA":            "VERACRUZ",
    "ORIZABA":            "VERACRUZ",
    "XALAPA":             "VERACRUZ",
    "COSOLEACAQUE":       "VERACRUZ",
    "COSAMALOAPAN":       "VERACRUZ",
    "ACAYUCAN":           "VERACRUZ",
    "NANCHITAL":          "VERACRUZ",
    "LAS CHOAPAS":        "VERACRUZ",
    "MARTINEZ DE LA TORRE":"VERACRUZ",
    "PEROTE":             "VERACRUZ",
    "TIERRA BLANCA":      "VERACRUZ",
    "COTAXTLA":           "VERACRUZ",
    "PAJARITOS":          "VERACRUZ",
    "TAD PAJARITOS":      "VERACRUZ",
    "PUERTO DE VERACRUZ": "VERACRUZ",
    # Nuevo Leon
    "MONTERREY":          "NUEVO LEON",
    "APODACA":            "NUEVO LEON",
    "CADEREYTA":          "NUEVO LEON",
    "GARCIA":             "NUEVO LEON",
    "GUADALUPE":          "NUEVO LEON",
    "SAN PEDRO GARZA GARCIA":"NUEVO LEON",
    "SAN NICOLAS DE LOS GARZA":"NUEVO LEON",
    "SANTA CATARINA":     "NUEVO LEON",
    "LINARES":            "NUEVO LEON",
    "CIENEGA DE FLORES":  "NUEVO LEON",
    # CDMX
    "AZCAPOTZALCO":       "CIUDAD DE MEXICO",
    "IZTAPALAPA":         "CIUDAD DE MEXICO",
    "GUSTAVO A. MADERO":  "CIUDAD DE MEXICO",
    "TLALPAN":            "CIUDAD DE MEXICO",
    "COYOACAN":           "COYOACAN",
    "XOCHIMILCO":         "CIUDAD DE MEXICO",
    "CUAJIMALPA":         "CIUDAD DE MEXICO",
    "ALVARO OBREGON":     "CIUDAD DE MEXICO",
    "MIGUEL HIDALGO":     "CIUDAD DE MEXICO",
    # Estado de Mexico
    "TOLUCA":             "ESTADO DE MEXICO",
    "ECATEPEC":           "ESTADO DE MEXICO",
    "NAUCALPAN":          "ESTADO DE MEXICO",
    "TLALNEPANTLA":       "ESTADO DE MEXICO",
    "ATIZAPAN":           "ESTADO DE MEXICO",
    "CUAUTITLAN":         "ESTADO DE MEXICO",
    "CUAUTITLAN IZCALLI": "ESTADO DE MEXICO",
    "TEPOTZOTLAN":        "ESTADO DE MEXICO",
    "TULTITLAN":          "ESTADO DE MEXICO",
    "COYOTEPEC":          "ESTADO DE MEXICO",
    "POLOTITLAN":         "ESTADO DE MEXICO",
    "ATLACOMULCO":        "ESTADO DE MEXICO",
    "ACAMBAY":            "ESTADO DE MEXICO",
    "JILOTEPEC":          "ESTADO DE MEXICO",
    "TEPALTITLAN":        "ESTADO DE MEXICO",
    "LERMA":              "ESTADO DE MEXICO",
    "NOPALTEPEC":         "ESTADO DE MEXICO",
    # Campeche
    "CIUDAD DEL CARMEN":  "CAMPECHE",
    "CD. DEL CARMEN":     "CAMPECHE",
    "CD DEL CARMEN":      "CAMPECHE",
    "CARMEN":             "CAMPECHE",
    "CALKINI":            "CAMPECHE",
    "PUERTO DE LERMA":    "CAMPECHE",
    # Tabasco
    "VILLAHERMOSA":       "TABASCO",
    "CARDENAS":           "TABASCO",
    "CUNDUACAN":          "TABASCO",
    "CENTRO":             "TABASCO",
    "PARAISO":            "TABASCO",
    "DOS BOCAS":          "TABASCO",
    # Jalisco
    "GUADALAJARA":        "JALISCO",
    "ZAPOPAN":            "JALISCO",
    "TLAQUEPAQUE":        "JALISCO",
    "PUERTO VALLARTA":    "JALISCO",
    "CD GUZMAN":          "JALISCO",
    "LAGOS DE MORENO":    "JALISCO",
    "ENCARNACION DIAZ":   "JALISCO",
    # Guanajuato
    "LEON":               "GUANAJUATO",
    "CELAYA":             "GUANAJUATO",
    "IRAPUATO":           "GUANAJUATO",
    "SALAMANCA":          "GUANAJUATO",
    "SILAO":              "GUANAJUATO",
    "APASEO EL GRANDE":   "GUANAJUATO",
    "SAN JOSE DE ITURBIDE":"GUANAJUATO",
    # Guerrero
    "BAHIA DE ACAPULCO":  "GUERRERO",
    "BAHIA DE ACA":       "GUERRERO", 
    "BAHIA DE A":         "GUERRERO", # <-- Parche para el recorte "BAHIA DE A" de la imagen
    "ACAPULCO":           "GUERRERO",
    "ZIHUATANEJO":        "GUERRERO",
    "CHILPANCINGO":       "GUERRERO",
    "IGUALA":             "GUERRERO",
    "TAXCO":              "GUERRERO",
    # Puebla
    "TEZIUTLAN":          "PUEBLA",
    "TEHUACAN":           "PUEBLA",
    "ATLIXCO":            "PUEBLA",
    "ACTAZINGO":          "PUEBLA",
    "SAN MARTIN TEXMELUCAN":"PUEBLA",
    # Queretaro
    "SANTIAGO DE QUERETARO":"QUERETARO",
    "SAN JUAN DEL RIO":   "QUERETARO",
    "PALMILLAS":          "QUERETARO",
    # Hidalgo
    "PACHUCA":            "HIDALGO",
    "TULA DE ALLENDE":    "HIDALGO",
    "TULA":               "HIDALGO",
    "ACTOPAN":            "HIDALGO",
    "SINGUILUCAN":        "HIDALGO",
    "SAN JUAN MICHIMALOYA":"HIDALGO",
    # Coahuila
    "SALTILLO":           "COAHUILA",
    "TORREON":            "COAHUILA",
    "MONCLOVA":           "COAHUILA",
    "PIEDRAS NEGRAS":     "COAHUILA",
    "RAMOS ARIZPE":       "COAHUILA",
    # San Luis Potosi
    "SAN LUIS POTOSI":    "SAN LUIS POTOSI",
    "MATEHUALA":          "SAN LUIS POTOSI",
    "CIUDAD VALLES":      "SAN LUIS POTOSI",
    "CD DEL MAIZ":        "SAN LUIS POTOSI",
    # Oaxaca
    "SALINA CRUZ":        "OAXACA",
    "JUCHITAN":           "OAXACA",
    "IXTEPEC":            "OAXACA",
    "TEHUANTEPEC":        "OAXACA",
    "MATIAS ROMERO":      "OAXACA",
    # Sinaloa
    "MAZATLAN":           "SINALOA",
    "CULIACAN":           "SINALOA",
    "LOS MOCHIS":         "SINALOA",
    "AHOME":              "SINALOA",
    "NAVOLATO":           "SINALOA",
    "CONCORDIA":          "SINALOA",
    "TOPOLOBAMPO":        "SINALOA",
    # Sonora
    "HERMOSILLO":         "SONORA",
    "CAJEME":             "SONORA",
    "CIUDAD OBREGON":     "SONORA",
    "GUAYMAS":            "SONORA",
    "NOGALES":            "SONORA",
    "EMPALME":            "SONORA",
    "CANANEA":            "SONORA",
    "NAVOJOA":            "SONORA",
    "IMURIS":             "SONORA",
    "AGUA PRIETA":        "SONORA",
    # Chihuahua
    "CIUDAD JUAREZ":      "CHIHUAHUA",
    "DELICIAS":           "CHIHUAHUA",
    "CUAUHTEMOC":         "CHIHUAHUA",
    "JIMENEZ":            "CHIHUAHUA",
    "CAMARGO":            "CHIHUAHUA",
    "PARRAL":             "CHIHUAHUA",
    # Chiapas
    "TUXTLA GUTIERREZ":   "CHIAPAS",
    "TAPACHULA":          "CHIAPAS",
    "ARRIAGA":            "CHIAPAS",
    "OCOZOCOAUTLA":       "CHIAPAS",
    "OCOZOCUAUTLA":       "CHIAPAS",
    # Morelos
    "CUERNAVACA":         "MORELOS",
    "CUAUTLA":            "MORELOS",
    # Yucatan
    "MERIDA":             "YUCATAN",
    "PROGRESO":           "YUCATAN",
    # Quintana Roo
    "CANCUN":             "QUINTANA ROO",
    "PLAYA DEL CARMEN":   "QUINTANA ROO",
    "CHETUMAL":           "QUINTANA ROO",
    # Baja California
    "TIJUANA":            "BAJA CALIFORNIA",
    "MEXICALI":           "BAJA CALIFORNIA",
    "ENSENADA":           "BAJA CALIFORNIA",
    "TECATE":             "BAJA CALIFORNIA",
    "LA RUMOROSA":        "BAJA CALIFORNIA",
    "ROSARITO":           "BAJA CALIFORNIA",
    # Baja California Sur
    "SAN JOSE DEL CABO":  "BAJA CALIFORNIA SUR",
    # Durango
    "LERDO":              "DURANGO",
    "GOMEZ PALACIO":      "DURANGO",
    "AGUILERA":           "DURANGO",
    "VICENTE GUERRERO":   "DURANGO",
    # Michoacan
    "LAZARO CARDENAS":    "MICHOACAN",
    "URUAPAN":            "MICHOACAN",
    "PATZCUARO":          "MICHOACAN",
    "COPANDARO DE GALEANA":"MICHOACAN",
    "COPANDARO":          "MICHOACAN",
    "CUITZEO":            "MICHOACAN",
    "NUEVA ITALIA":       "MICHOACAN",
    # Otros
    "SAYULA DE ALEMAN":   "VERACRUZ",
    "TEPIC":              "NAYARIT",
    "CALPULALPAN":        "TLAXCALA",
    "CONCEPCION DEL ORO": "ZACATECAS",
    "FRESNILLO":          "ZACATECAS",
    "COSIO":              "AGUASCALIENTES",
    "COLON":              "QUERETARO",
    "MADERO":             "TAMAULIPAS",
}

# ============================================================
# CATALOGO 4: Internacionales
# ============================================================
INTERNACIONAL_KEYWORDS = {
    "USA":            ("NO ESPECIFICADO", "USA"),
    "EUA":            ("NO ESPECIFICADO", "USA"),
    "EEUU":           ("NO ESPECIFICADO", "USA"),
    "UNITED STATES":  ("NO ESPECIFICADO", "USA"),
    "TEXAS":          ("TEXAS", "USA"),
    "TX":             ("TEXAS", "USA"),
    "HOUSTON":        ("TEXAS", "USA"),
    "GALVESTON":      ("TEXAS", "USA"),
    "BOLIVAR ROADS":  ("TEXAS", "USA"),
    "BEAUMONT":       ("TEXAS", "USA"),
    "PORT ARTHUR":    ("TEXAS", "USA"),
    "PORT ARTUR":     ("TEXAS", "USA"),
    "PASADENA":       ("TEXAS", "USA"),
    "LOUISIANA":      ("LOUISIANA", "USA"),
    "LA":             ("LOUISIANA", "USA"),
    "LAKE CHARLES":   ("LOUISIANA", "USA"),
    "TERMINAL DE ST. CHARLES VALERO": ("LOUISIANA", "USA"),
    "CALIFORNIA":     ("CALIFORNIA", "USA"),
    "CALIFORNIA(USA)":("CALIFORNIA", "USA"),
    "SAN FRANCISCO":  ("CALIFORNIA", "USA"),
    "SAN DIEGO":      ("CALIFORNIA", "USA"),
    "NUEVA JERSEY":   ("NEW JERSEY", "USA"),
    "NEW JERSEY":     ("NEW JERSEY", "USA"),
    "PANAMA":         ("PANAMA", "PANAMA"),
    "BALBOA":         ("PANAMA", "PANAMA"),
    "GATUN":          ("PANAMA", "PANAMA"),
    "CHILE":          ("NO ESPECIFICADO", "CHILE"),
    "TALCAHUANO":     ("BIOBIO", "CHILE"),    
    "BAHIA CONCEPCION":("BIOBIO", "CHILE"),   
    "ALEMANIA":       ("NO ESPECIFICADO", "ALEMANIA"),
    "SINGAPORE":      ("SINGAPUR", "SINGAPUR"),
    "SINGAPUR":       ("SINGAPUR", "SINGAPUR"),   
    "AMSTERDAM":      ("HOLANDA SEPTENTRIONAL", "PAISES BAJOS"),
    "PLATAFORMA BICENTENARIO DE LA CIA EN EL POZO TRION-1": ("GOLFO DE MEXICO", "MAR INTERNACIONAL"),
    "AGUAS DEL GOLFO DE MEXICO, REGION MARINA NOROESTE, ACTIVO DE PRODUCCION CANTARELL": ("GOLFO DE MEXICO", "MAR INTERNACIONAL"),
    "REGION MARINA":  ("GOLFO DE MEXICO", "MAR INTERNACIONAL"),
    "GOLFO DE MEXICO":("GOLFO DE MEXICO", "MAR INTERNACIONAL"),
    "PACIFICO NORTE": ("PACIFICO NORTE", "MAR INTERNACIONAL"),
    "AGUAS DEL GOLFO":("GOLFO DE MEXICO", "MAR INTERNACIONAL"),
    "LUGAR/TERMINAL: PACIFIC AREA LIGHTERING":("PACIFICO NORTE", "MAR INTERNACIONAL"),
    "BAHIA DE CAMPECHE":("GOLFO DE MEXICO", "MAR INTERNACIONAL"),
    "NOHOCH":          ("GOLFO DE MEXICO", "MAR INTERNACIONAL"),
    "CANTARELL":       ("GOLFO DE MEXICO", "MAR INTERNACIONAL"),
}

CAPITALES_POR_PAIS = {
    "PANAMA":        "CIUDAD DE PANAMA",
    "CHILE":         "SANTIAGO",
    "ALEMANIA":      "BERLIN",
    "SINGAPUR":      "SINGAPUR",
    "PAISES BAJOS":  "AMSTERDAM",
}

# ============================================================
# CATALOGO 5: Autopistas/Rutas conocidas
# ============================================================
AUTOPISTAS = {
    "MEX-QRO":                      "QUERETARO",
    "MEXICO-QUERETARO":             "QUERETARO",
    "MEXICO QRO":                   "QUERETARO",
    "ARCO NORTE":                   "HIDALGO",
    "CORDOBA-VERACRUZ":             "VERACRUZ",
    "CORDOBA-PUEBLA":               "VERACRUZ",
    "COATZACOALCOS-VILLAHERMOSA":  "VERACRUZ",
    "CUERNAVACA-IGUALA":           "MORELOS",
    "CUERNAVACA-ACAPULCO":         "GUERRERO",
    "ACAPULCO-ZIHUATANEJO":        "GUERRERO",
    "LEON-SALAMANCA":              "GUANAJUATO",
    "LEON-AGUASCALIENTES":         "JALISCO",
    "SALTILLO-MONTERREY":          "COAHUILA",
    "MONTERREY-CD VICTORIA":       "NUEVO LEON",
    "PIEDRAS NEGRAS-NUEVO LAREDO": "COAHUILA",
    "TORREON-DURANGO":             "DURANGO",
    "MAZATLAN-DURANGO":            "SINALOA",
    "MOCHIS-NAVOJOA":              "SINALOA",
    "TIJUANA-MEXICALI":            "BAJA CALIFORNIA",
    "TIJUANA-TECATE":              "BAJA CALIFORNIA",
    "LAZARO CARDENAS-URUAPAN":     "MICHOACAN",
    "PATZCUARO-URUAPAN":           "MICHOACAN",
    "CIUDAD VALLES-TAMPICO":       "SAN LUIS POTOSI",
    "CIUDAD VICTORIA-MONTERREY":   "TAMAULIPAS",
    "ARRIAGA-OCOZOCOAUTLA":        "CHIAPAS",
    "ARRIAGA-OCOZOCUAUTLA":        "CHIAPAS",
    "SALINA CRUZ-MATIAS ROMERO":   "OAXACA",
    "MITLA-TEHUANTEPEC":           "OAXACA",
    "TINAJA-ACAYUCAN":             "VERACRUZ",
    "MEX-TUXPAN":                  "VERACRUZ",
    "SAN LUIS POTOSI-QUERETARO":   "SAN LUIS POTOSI",
    "PALMILLAS-APASEO":            "QUERETARO",
    "TEPIC-PUENTE TALISMAN":       "NAYARIT",
    "APODACA-MONTERREY":           "NUEVO LEON",
    "SANTA CATALINA-NUEVA ITALIA": "MICHOACAN",
    "CUITZEO-MORELIA":             "MICHOACAN",
    "Carretera Federal 40":         "SINALOA",
    "Carretera México 40":         "SINALOA",
}

# ============================================================
# FUNCION DE NORMALIZACION
# ============================================================
def normalizar(texto):
    if pd.isna(texto) or str(texto).strip() in ("", "-", "nan", "N/A"):
        return ""
    t = unidecode(str(texto)).upper().strip()
    t = re.sub(r"[;:'\"\[\]\(\)\{\}*#!]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# ============================================================
# FUNCION PRINCIPAL CORREGIDA
# ============================================================
def clasificar_estado(location_raw):
    texto = normalizar(location_raw)

    if not texto:
        return {"estado": "NO ESPECIFICADO", "pais": "NO ESPECIFICADO", "metodo": "vacio", "confianza": "N/A"}

    # -- ETAPA 1: Internacionales --
    for keyword, info_pais in sorted(INTERNACIONAL_KEYWORDS.items(), key=lambda x: len(x[0]), reverse=True):
        kw_clean = unidecode(keyword).upper()

        # Filtros de guardrail para evitar falsos positivos
        if kw_clean == "LA" and not re.search(r'\b(USA|EUA|EEUU|UNITED STATES)\b', texto):
            continue

        if kw_clean == "CALIFORNIA" and not re.search(r'\b(USA|EUA|EEUU|UNITED STATES)\b', texto):
            continue

        if kw_clean == "BAHIA DE CAMPECHE":
            if "CAMPECHE" in texto and not re.search(r'\b(REGION MARINA|GOLFO|PLATAFORMA|ACTIVO)\b', texto):
                continue

        # ¡AQUÍ ESTÁ LA CORRECCIÓN DE INDENTACIÓN! Todo va adentro del bloque de coincidencia exitosa
        if re.search(r'\b' + re.escape(kw_clean) + r'\b', texto):
            texto_sin_golfo = re.sub(r'GOLFO\s+DE\s+MEXICO', '', texto)
            menciona_mexico = bool(re.search(r'\b(MEXICO|MEX)\b', texto_sin_golfo))
            tiene_estado_mx = any(est_name in texto for est_name in ESTADOS_MX)

            if menciona_mexico or tiene_estado_mx:
                continue  # Salta si es en realidad un caso de México

            estado_assigned = info_pais[0]
            pais_asignado   = info_pais[1]
            
            if estado_assigned == "NO ESPECIFICADO":
                estado_assigned = CAPITALES_POR_PAIS.get(pais_asignado, "NO ESPECIFICADO")
                
            return {
                "estado":    estado_assigned,
                "pais":      pais_asignado,
                "metodo":    "internacional",
                "confianza": "ALTA"
            }

    # -- ETAPA 2: Estados explícitos en el texto --
    for est_clean, est_oficial in ESTADOS_MX.items():
        if est_clean in texto:
            return {"estado": est_oficial, "pais": "MEXICO", "metodo": "estado_explicito", "confianza": "ALTA"}

    # -- ETAPA 3: Abreviaturas estrictas --
    for abrev in sorted(ABREVIATURAS.keys(), key=len, reverse=True):
        abrev_clean = unidecode(abrev).upper()
        pattern = r'\b' + re.escape(abrev_clean) + r'\b'
        if re.search(pattern, texto):
            return {"estado": ABREVIATURAS[abrev], "pais": "MEXICO", "metodo": "abreviatura", "confianza": "ALTA"}

    # -- ETAPA 4: Autopistas --
    for ruta, estado in AUTOPISTAS.items():
        ruta_clean = unidecode(ruta).upper()
        if ruta_clean in texto:
            return {"estado": estado, "pais": "MEXICO", "metodo": "autopista", "confianza": "MEDIA"}

    # -- ETAPA 5: Ciudades/Municipios --
    mejor_match = None
    mejor_len   = 0
    for ciudad, estado in CIUDADES.items():
        ciudad_clean = unidecode(ciudad).upper()
        if ciudad_clean in texto and len(ciudad_clean) > mejor_len:
            mejor_match = (ciudad, estado)
            mejor_len   = len(ciudad_clean)

    if mejor_match:
        return {"estado": mejor_match[1], "pais": "MEXICO", "metodo": "ciudad:" + mejor_match[0], "confianza": "MEDIA"}

    # -- ETAPA 6: Infraestructura genérica sin estado --
    if "TERMINAL DE PEMEX" in texto or "REF.RC-R-204-2010" in texto:
        return {"estado": "NO ESPECIFICADO", "pais": "MEXICO", "metodo": "infraestructura_pemex", "confianza": "BAJA"}

    return {"estado": "NO ESPECIFICADO", "pais": "NO ESPECIFICADO", "metodo": "sin_match", "confianza": "N/A"}

# ============================================================
# APLICACIÓN AL DATAFRAME
# ============================================================
def agregar_estado(df, col_location="LOCATION"):
    resultados = df[col_location].apply(clasificar_estado)
    
    df["ESTADO"]        = resultados.apply(lambda r: r["estado"])
    df["PAIS"]          = resultados.apply(lambda r: r["pais"])
    df["METODO_MATCH"]  = resultados.apply(lambda r: r["metodo"])
    df["CONFIANZA"]     = resultados.apply(lambda r: r["confianza"])
    return df

# Aplicar la función al DataFrame
df = agregar_estado(df, col_location="LOCATION")
# Analizar resultados
print("Distribución de PAISES:")
print(df["PAIS"].value_counts()) 



Distribución de PAISES:
PAIS
MEXICO               2542
NO ESPECIFICADO       311
USA                    17
PANAMA                  6
CHILE                   2
MAR INTERNACIONAL       2
PAISES BAJOS            1
ALEMANIA                1
SINGAPUR                1
Name: count, dtype: int64


### 3.3 Tratamiento de valores duplicados


In [113]:
# ================================
# MANEJO DE LOS DUPLICADOS
# ================================
#region DUPLICADOS
id_col = "CLAIM NUMBER"

def rellenar_informacion_grupo(grupo):
    """
    Completa LOCATION, DESCRIPTION, ESTADO y PAIS faltantes
    usando datos del mismo siniestro (otra fila del mismo CLAIM NUMBER).
    """
    locations_validas = grupo[grupo["LOCATION"] != "NO ESPECIFICADO"]["LOCATION"].unique()
    if len(locations_validas) > 0:
        grupo.loc[grupo["LOCATION"] == "NO ESPECIFICADO", "LOCATION"] = locations_validas[0]

    descriptions_validas = grupo[grupo["DESCRIPTION"] != "NO ESPECIFICADO"]["DESCRIPTION"].unique()
    if len(descriptions_validas) > 0:
        grupo.loc[grupo["DESCRIPTION"] == "NO ESPECIFICADO", "DESCRIPTION"] = descriptions_validas[0]

    estados_validos = grupo[grupo["ESTADO"] != "NO ESPECIFICADO"]["ESTADO"].unique()
    if len(estados_validos) > 0:
        grupo.loc[grupo["ESTADO"] == "NO ESPECIFICADO", "ESTADO"] = estados_validos[0]

    # ✅ FIX: Heredar PAIS desde otra fila del mismo siniestro
    if "PAIS" in grupo.columns:
        paises_validos = grupo[~grupo["PAIS"].isin(["NO ESPECIFICADO"])]["PAIS"].unique()
        if len(paises_validos) > 0:
            grupo.loc[grupo["PAIS"] == "NO ESPECIFICADO", "PAIS"] = paises_validos[0]

    return grupo

def unir_no_especificado(grupo):
    if len(grupo) == 2:
        fila_buena = grupo[
            (grupo["LOCATION"] != "NO ESPECIFICADO") &
            (grupo["GROSS RESERVE"] > 0) &
            (grupo["DEDUCTIBLE"] > 0) &
            (grupo["DESCRIPTION"] != "NO ESPECIFICADO")
        ]
        if len(fila_buena) == 1:
            return fila_buena
    return grupo

num_cols = [
    'Cumulative EXPENSES PAID',
    'Cumulative VALUATION EXPENSES',
    'Cumulative CLAIMS PAID',
    'Total Claims Paid Inward',
    'Total Claims Paid no Alae',
    'Inward Incurred',
    'Inward Incurred no Alae',
    'GROSS RESERVE',
    'DEDUCTIBLE'
]

def seleccionar_mejores_registros(grupo):
    grupo = grupo.copy()

    # 0️⃣ PAGOS REALES GANAN TODO
    if "Cumulative CLAIMS PAID" in grupo.columns and len(grupo) > 1:
        con_paid = grupo[grupo["Cumulative CLAIMS PAID"].fillna(0) > 0]
        sin_paid = grupo[grupo["Cumulative CLAIMS PAID"].fillna(0) == 0]

        if not con_paid.empty and not sin_paid.empty:
            ded_con_paid = set(con_paid["DEDUCTIBLE"].fillna(0).unique())
            sin_paid_con_ded_diff = sin_paid[
                (~sin_paid["DEDUCTIBLE"].fillna(0).isin(ded_con_paid)) &
                (sin_paid["DEDUCTIBLE"].fillna(0) > 0)
            ]
            return pd.concat([con_paid, sin_paid_con_ded_diff])
        elif not con_paid.empty:
            return con_paid

    # 0️⃣.5️⃣ CASO ESPECIAL: STATUS P vs C (Prevalece el registro C)
    if "STATUS" in grupo.columns and len(grupo) > 1:
        fila_p = grupo[grupo["STATUS"] == "P"]
        fila_c = grupo[grupo["STATUS"] == "C"]

        if len(fila_p) == 1 and len(fila_c) == 1:
            cols_gastos = [
                "Cumulative EXPENSES PAID",
                "Cumulative VALUATION EXPENSES",
                "Cumulative CLAIMS PAID",
                "Total Claims Paid Inward",
                "Total Claims Paid no Alae",
                "Inward Incurred",
                "Inward Incurred no Alae"
            ]
            cols_presentes = [c for c in cols_gastos if c in grupo.columns]
            if cols_presentes:
                valores_gastos = fila_p[cols_presentes].fillna(0).iloc[0]
                if (valores_gastos > 0).any():
                    for col in cols_presentes:
                        if fila_c[col].fillna(0).values[0] == 0:
                            grupo.loc[grupo["STATUS"] == "C", col] = valores_gastos[col]
            grupo = grupo[grupo["STATUS"] == "C"]

    # 1️⃣ DEDUCIBLE
    if "DEDUCTIBLE" in grupo.columns and len(grupo) > 1:
        cols_compare = [c for c in grupo.columns if c not in ["DEDUCTIBLE", "STATUS", "STATUS_ORIGINAL", "Nat Cat", "MES CARGA"]]
        dup = grupo.duplicated(subset=cols_compare, keep=False)
        if dup.any():
            candidatos = grupo[dup]
            con_ded = candidatos[candidatos["DEDUCTIBLE"].fillna(0) > 0]
            if not con_ded.empty:
                grupo = con_ded[con_ded["DEDUCTIBLE"] == con_ded["DEDUCTIBLE"].max()]

    # 2️⃣ DEFINIR FINANZAS
    _num_cols = grupo.select_dtypes(include="number").columns
    tiene_finanzas_fila = grupo[_num_cols].fillna(0).sum(axis=1) > 0
    ambos_tienen_finanzas = tiene_finanzas_fila.all()

    # 3️⃣ PAGOS
    cols_pagos = [c for c in ["Cumulative EXPENSES PAID", "Cumulative VALUATION EXPENSES",
                               "Cumulative CLAIMS PAID"] if c in grupo.columns]
    tiene_reserve_o_ded = (
        (grupo["DEDUCTIBLE"].fillna(0) > 0) | (grupo["GROSS RESERVE"].fillna(0) > 0)
    ).any()

    if cols_pagos and len(grupo) > 1 and not tiene_reserve_o_ded:
        tiene_pagos = grupo[cols_pagos].fillna(0).gt(0).any(axis=1)
        if tiene_pagos.any():
            grupo = grupo[tiene_pagos]

    # 4️⃣ ELIMINAR STATUS = C SIN RESERVE NI DED
    if len(grupo) > 1 and ambos_tienen_finanzas:
        eliminar = (
            (grupo["GROSS RESERVE"].fillna(0) == 0) &
            (grupo["DEDUCTIBLE"].fillna(0) == 0) &
            (grupo["STATUS"] == "C")
        )
        if eliminar.any() and (~eliminar).any():
            grupo = grupo[~eliminar]

    # 5️⃣ STATUS P
    if "STATUS" in grupo.columns and len(grupo) > 1 and not ambos_tienen_finanzas:
        status_p = grupo[grupo["STATUS"] == "P"]
        if not status_p.empty:
            grupo = status_p

    # 6️⃣ FILAS CON NÚMEROS
    if len(_num_cols) > 0 and len(grupo) > 1:
        con_numeros = grupo[_num_cols].notna().any(axis=1)
        if con_numeros.any():
            grupo = grupo[con_numeros]

    # 7️⃣ COMPLETITUD
    completitud = grupo.notna().mean(axis=1)
    grupo = grupo.loc[completitud == completitud.max()]

    # 8️⃣ DESEMPATE FINAL
    if len(grupo) > 1 and len(_num_cols) > 0:
        nums = grupo[_num_cols].fillna(0)
        if nums.nunique().max() == 1:
            grupo = grupo.sort_index().tail(1)

    return grupo

# ================================
# APLICAR LOS TRES GROUPBY
# ================================
def procesar_grupo(grupo):
    grupo = rellenar_informacion_grupo(grupo)
    grupo = unir_no_especificado(grupo)
    grupo = seleccionar_mejores_registros(grupo)
    return grupo

df = (
    df.set_index(id_col)
    .groupby(level=0, group_keys=False)
    .apply(procesar_grupo)
    .reset_index()
)

print(f"✓ {id_col} presente: {id_col in df.columns}")

# ================================
# ✅ PARCHE DE SEGURIDAD: inferir PAIS desde ESTADO
# Cubre casos donde rellenar_informacion_grupo heredó ESTADO
# de otra fila pero PAIS quedó como "NO ESPECIFICADO"
# ================================
if "PAIS" in df.columns and "ESTADO" in df.columns:
    ESTADOS_MEXICO = set(ESTADOS_MX.values())
    mask_sin_pais   = df["PAIS"] == "NO ESPECIFICADO"
    mask_estado_mx  = df["ESTADO"].isin(ESTADOS_MEXICO)
    n_corregidos    = (mask_sin_pais & mask_estado_mx).sum()
    df.loc[mask_sin_pais & mask_estado_mx, "PAIS"] = "MEXICO"
    if n_corregidos > 0:
        print(f"✅ PAIS corregido a MEXICO en {n_corregidos} registros (ESTADO era México pero PAIS era NO ESPECIFICADO)")

# ================================
# ELIMINAR FILAS CON TODOS CEROS
# ================================
num_cols = [
    'Cumulative EXPENSES PAID',
    'Cumulative VALUATION EXPENSES',
    'Cumulative CLAIMS PAID',
    'Total Claims Paid Inward',
    'Total Claims Paid no Alae',
    'Inward Incurred',
    'Inward Incurred no Alae',
    'GROSS RESERVE',
    'DEDUCTIBLE'
]

id_col = "CLAIM NUMBER"

# --- Paso 1: Identificar filas con todos ceros ---
mask_ceros = df[num_cols].fillna(0).sum(axis=1) == 0

# --- Paso 2: Claims con mas de 1 fila ---
filas_por_claim = df.groupby(id_col)[id_col].transform('size')
tiene_multiples_filas = filas_por_claim > 1

# --- Paso 3: Normalizar COVER para comparacion ---
df["_COVER_NORM"] = (
    df["COVER"]
    .fillna("NO ESPECIFICADO")
    .astype(str)
    .str.upper()
    .str.strip()
)

# --- Paso 4: Verificar si la otra fila del claim tiene el mismo COVER ---
def misma_cover_en_grupo(grupo):
    resultado = pd.Series(False, index=grupo.index)
    covers_unicos = grupo["_COVER_NORM"].unique()
    if len(covers_unicos) == 1:
        resultado[:] = True
    else:
        for idx, row in grupo.iterrows():
            cover_actual = row["_COVER_NORM"]
            otras_con_mismo_cover = grupo[(grupo.index != idx) & (grupo["_COVER_NORM"] == cover_actual)]
            if len(otras_con_mismo_cover) > 0:
                resultado[idx] = True
    return resultado

misma_cover = (
    df.groupby(id_col, group_keys=False)
    .apply(misma_cover_en_grupo)
)

# --- Paso 5: Mascara final de eliminacion ---
eliminar = mask_ceros & tiene_multiples_filas & misma_cover

# --- Paso 6: Log de auditoria ---
n_eliminadas = eliminar.sum()
if n_eliminadas > 0:
    print(f"\n{'='*60}")
    print(f"ELIMINACION DE FILAS CON TODOS CEROS")
    print(f"{'='*60}")
    print(f"Filas con todos ceros:           {mask_ceros.sum()}")
    print(f"  - En claims con 1 sola fila:   {(mask_ceros & ~tiene_multiples_filas).sum()} (se conservan)")
    print(f"  - En claims con 2+ filas:      {(mask_ceros & tiene_multiples_filas).sum()}")
    print(f"    - Mismo COVER (eliminables):  {n_eliminadas}")
    print(f"    - COVER diferente (protegidas):{(mask_ceros & tiene_multiples_filas & ~misma_cover).sum()}")
    eliminadas = df[eliminar][[id_col, "COVER", "STATUS"]].copy()
    for _, row in eliminadas.iterrows():
        print(f"  CLAIM {row[id_col]}: COVER={row['COVER']}, STATUS={row['STATUS']}")
else:
    print("No hay filas con ceros candidatas a eliminacion.")

# --- Paso 7: Eliminar y limpiar ---
df = df[~eliminar].copy()
df = df.drop(columns=["_COVER_NORM"])

# --- Paso 8: drop_duplicates CON subset explicito ---
antes_dedup = len(df)
cols_dedup = [c for c in df.columns if c not in ["MES CARGA"]]
df = df.drop_duplicates(subset=cols_dedup)
despues_dedup = len(df)
if antes_dedup != despues_dedup:
    print(f"\nDuplicados exactos eliminados: {antes_dedup - despues_dedup}")

print(f"\nRegistros finales: {len(df)}")

# ================================
# REORDENAR COLUMNAS: CLAIM NUMBER después de CLAIMS-ID
# ================================
cols = df.columns.tolist()
if 'CLAIM NUMBER' in cols and 'CLAIMS-ID' in cols:
    cols.remove('CLAIM NUMBER')
    idx = cols.index('CLAIMS-ID')
    cols.insert(idx + 1, 'CLAIM NUMBER')
    df = df[cols]
    print("✓ CLAIM NUMBER movido después de CLAIMS-ID")
#endregion

✓ CLAIM NUMBER presente: True
No hay filas con ceros candidatas a eliminacion.

Registros finales: 2854
✓ CLAIM NUMBER movido después de CLAIMS-ID


## 4. Exportación de resultados


In [114]:
# Guardar Excel
# Definición de rutas (Asegúrate de que las variables estén declaradas previamente)
ruta_salida2 = os.path.join(carpeta_salida2, nombre_archivo_salida)

# ================================
# VALIDACIÓN PRE-EXPORTACIÓN
# ================================
print(f"\n=== VALIDACIÓN FINAL ===")
print(f"Total registros: {len(df)}")

# Evitar KeyError si la columna no existe aún en el flujo
if "CLAIM NUMBER" in df.columns:
    print(f"Claims únicos: {df['CLAIM NUMBER'].nunique()}")
else:
    print("ALERTA: La columna 'CLAIM NUMBER' no existe en el DataFrame.")

print(f"\nDistribución STATUS:")
print(df["STATUS"].value_counts())

print(f"\nDistribución COVER:")
print(df["COVER"].value_counts())

# Corregido: Apunta a STATUS que es la columna homologada
registros_sin_status = df["STATUS"].isin(["NO ESPECIFICADO", "NAN"]).sum()
print(f"\nRegistros sin STATUS (No especificado): {registros_sin_status}")

# Verificar NaN o valores "vacíos de negocio" en columnas clave
columnas_clave = ["STATUS", "COVER"]
if "CLAIM NUMBER" in df.columns:
    columnas_clave.append("CLAIM NUMBER")

for col in columnas_clave:
    # Cuenta nulos reales de Pandas
    nulos_reales = df[col].isna().sum()
    # Cuenta nulos simulados por la conversión a string
    nulos_texto = df[col].astype(str).str.upper().isin(["NAN", "NO ESPECIFICADO", ""]).sum()
    
    if nulos_reales > 0 or nulos_texto > 0:
        print(f"⚠️ ALERTA: '{col}' tiene {nulos_reales} NaN reales y {nulos_texto} valores vacíos/sin especificar.")

# ================================
# EXPORTACIÓN A EXCEL (xlsxwriter)
# ================================
with pd.ExcelWriter(ruta_salida2, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Hoja1", index=False)
    
    workbook  = writer.book
    worksheet = writer.sheets["Hoja1"]
    
    # Formato porcentaje sin decimales
    formato_pct = workbook.add_format({'num_format': '0%'})
    
    # Aplicar formato a KOT SHARE si existe
    if "KOT SHARE" in df.columns:
        col_idx = df.columns.get_loc("KOT SHARE")
        # Solución al bug: Se asigna un ancho explícito (15) para evitar que la columna se oculte o rompa
        worksheet.set_column(col_idx, col_idx, 15, formato_pct)
    else:
        print("AVISO: No se encontró la columna 'KOT SHARE', omitiendo formato de porcentaje.")

print(f"\n✅ Archivo guardado exitosamente en: {ruta_salida2}")


=== VALIDACIÓN FINAL ===
Total registros: 2854
Claims únicos: 2853

Distribución STATUS:
STATUS
P    1916
C     778
T     160
Name: count, dtype: int64

Distribución COVER:
COVER
CARGO                              1663
CARGA POLIETILENO                   887
P&I                                 117
RC FLETADORES(PEMEX)                 75
CASCO Y MAQ.                         61
FLETADORES(PMI)                      43
NO ESPECIFICADO                       4
JACK-UPS(DAÑO FISICO)                 3
EQUIPO FERROVIARIO(DAÑO FÍSICO)       1
Name: count, dtype: int64

Registros sin STATUS (No especificado): 0
⚠️ ALERTA: 'COVER' tiene 0 NaN reales y 4 valores vacíos/sin especificar.

✅ Archivo guardado exitosamente en: C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/2025/202512\202512_Siniestros_Marine_PROCESADO.xlsx
